#1. Khai báo thư viện



In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from sklearn.metrics import mean_absolute_error
import pickle

drive.mount('/content/drive')
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/EDA")


#2. Đọc tập tin

In [ ]:
# Đường dẫn
folder_train_val = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/Data_Train_Val_Test")

file_path_train = folder_train_val / "Train.csv"
file_path_val = folder_train_val / "Validation.csv"
file_path_test= folder_train_val / "Test.csv"

df_train=pd.read_csv(file_path_train)
df_val=pd.read_csv(file_path_val)
df_test=pd.read_csv(file_path_test)

print("in 5 dòng đầu\n", df_train)
print("tập train\n", df_train)


#3. Thiết lập cấu hình và huấn luyện mô hình

##3.1 Sáu cấu hình phổ biến

In [ ]:
configurations = [
    {
        'error': 'add',
        'trend': None,
        'seasonal': None,
        'damped_trend': False,
        'name': '1. Simple Exponential Smoothing (SES)',
    },
    {
        'error': 'add',
        'trend': 'add',
        'seasonal': None,
        'damped_trend': False,
        'name': "2. Holt's Linear Trend",
    },
    {
        'error': 'add',
        'trend': 'add',
        'seasonal': None,
        'damped_trend': True,
        'name': '3. Additive Damped Trend',
    },
    {
        'error': 'add',
        'trend': 'add',
        'seasonal': 'add',
        'damped_trend': False,
        'name': "4. Additive Holt-Winters'",
    },
    {
        'error': 'mul',
        'trend': 'add',
        'seasonal': 'mul',
        'damped_trend': False,
        'name': "5. Multiplicative Holt-Winters'",
    },
    {
        'error': 'add',
        'trend': 'add',
        'seasonal': 'add',
        'damped_trend': True,
        'name': "6. Damped Holt-Winters'",
    },
]


##3.2 Huấn luyện mô hình

In [ ]:
# 1. Chuẩn hóa dữ liệu
y_train = df_train['TEU'].astype(float).reset_index(drop=True)
y_val = df_val['TEU'].astype(float).reset_index(drop=True)
y_test = df_test['TEU'].astype(float).reset_index(drop=True)

results = []
errors = []

# 2. Huấn luyện và đánh giá các cấu hình ETS

for config in configurations:
    try:
        # Fit mô hình trên tập Train
        model = ETSModel(
            y_train,
            error=config['error'],
            trend=config['trend'],
            seasonal=config['seasonal'],
            damped_trend=config['damped_trend'],
            seasonal_periods=(
                12 if config['seasonal'] is not None else None
            )
        )

        fitted = model.fit(disp=False)

        # Dự báo trên tập Validation
        val_forecast = np.asarray(
            fitted.forecast(steps=len(y_val))
        ).reshape(-1)

        # Sai số: giá trị thực tế - giá trị dự báo
        forecast_errors = y_val.to_numpy() - val_forecast


        # MEA: Sai số trung bình
        me = np.mean(abs(forecast_errors))

        # SDAE mẫu của sai số
        std_error = (
            np.std(forecast_errors, ddof=1)
            if len(forecast_errors) > 1
            else np.nan
        )

        # Lưu kết quả
        results.append({
            'Model Name': config['name'],
            'AIC': fitted.aic,
            'AICc': fitted.aicc,
            'BIC': fitted.bic,
            'MAE': me,
            'STD MAE': std_error,
            'Fitted Model Object': fitted
        })

    except Exception as e:
        errors.append({
            'Model Name': config.get('name', 'Unknown'),
            'Error': str(e)
        })

# 3. Tổng hợp và xếp hạng mô hình
results_df = pd.DataFrame(results)

if not results_df.empty:

    results_df = results_df.sort_values(
        by='AICc',
        ascending=True
    ).reset_index(drop=True)

    display_cols = [
        'Model Name',
        'AIC',
        'AICc',
        'BIC',
        'MAE',
        'STD MAE'
    ]

    print('=== BẢNG SO SÁNH MÔ HÌNH ETS ===')
    print(results_df[display_cols].to_string(index=False))


##3.3 Lựa chọn cấu hình

In [ ]:
# Tìm Phương pháp làm trơn hàm mũ Holt-Winter tắt dần trong kết quả
best_row = results_df[
    results_df['Model Name'].str.contains(
        'Damped Holt-Winters',
        case=False,
        regex=True
    )
].iloc[0]

best_config_name = best_row['Model Name']
best_model = best_row['Fitted Model Object']

print(f'\nMô hình được lựa chọn: {best_config_name}')

# In cấu hình mô hình
best_config = next(
    config for config in configurations
    if config['name'] == best_config_name
)

print('\n=== THAM SỐ CẤU HÌNH MÔ HÌNH Damped Holt-Winters ===')
for key, value in best_config.items():
    if key != 'name':
        print(f'{key}: {value}')

# In tham số ước lượng
print('\n=== CÁC THAM SỐ ƯỚC LƯỢNG ===')
print(best_model.params)

# 5. Vẽ biểu đồ so sánh 5 chỉ số của 6 mô hình ETS
metrics = ['BIC', 'AIC', 'AICc', 'MAE', 'STD MAE']

plot_df = results_df.copy()

# Số lượng mô hình và chỉ số
n_models = len(plot_df)
n_metrics = len(metrics)

x = np.arange(n_models)
width = 0.15

# Khởi tạo biểu đồ
fig, ax = plt.subplots(figsize=(18, 8))

# 5 màu tương ứng 5 chỉ số
colors = [
    'steelblue',
    'darkorange',
    'seagreen',
    'mediumpurple',
    'indianred'
]

# Vẽ 5 cột cho mỗi mô hình
for i, metric in enumerate(metrics):

    offset = (i - (n_metrics - 1) / 2) * width

    bars = ax.bar(
        x + offset,
        plot_df[metric],
        width,
        label=metric,
        color=colors[i]
    )

    # Hiển thị giá trị trên từng cột
    for bar in bars:
        height = bar.get_height()

        ax.annotate(
            f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=7,
            rotation=90
        )

# Tên mô hình trên trục X
ax.set_xticks(x)
ax.set_xticklabels(
    plot_df['Model Name'],
    rotation=25,
    ha='right',
    fontsize=10
)

# Tiêu đề và nhãn
ax.set_title(
    'SO SÁNH 5 CHỈ SỐ ĐÁNH GIÁ CÁC MÔ HÌNH ETS',
    fontsize=15,
    fontweight='bold',
    pad=20
)

ax.set_xlabel('Mô hình ETS', fontsize=12)
ax.set_ylabel('Giá trị chỉ số', fontsize=12)

# Chú thích
ax.legend(
    title='Chỉ số đánh giá',
    loc='upper left',
    bbox_to_anchor=(1.01, 1)
)

ax.grid(
    axis='y',
    linestyle='--',
    alpha=0.3
)

plt.tight_layout()
plt.show()

#4. Dự báo trên tập test

In [ ]:
# 1. Gộp tập train và validation để huấn luyện lại mô hình
y_train_val = pd.concat(
    [df_train['TEU'], df_val['TEU']],
    ignore_index=True
).astype(float)

#2. Huấn luyện lại mô hình
model_final = ETSModel(
    y_train_val,
    error=best_config['error'],
    trend=best_config['trend'],
    seasonal=best_config['seasonal'],
    damped_trend=best_config['damped_trend'],
    seasonal_periods=(
        12 if best_config['seasonal'] is not None else None
    )
)

fitted_final = model_final.fit(disp=False)

print('=== MÔ HÌNH SAU KHI REFIT ===')
print(fitted_final.params)

# 3. Dự báo trên tập tets
y_dudoan = np.asarray(
    fitted_final.forecast(steps=len(y_test))
).reshape(-1)

y_thucte = y_test.to_numpy()

# 4. Tính toán các chỉ số đánh giá
# Sai số tuyệt đối
errors_abs = np.abs(y_thucte - y_dudoan)

# MAE
mae = np.mean(errors_abs)

# Độ lệch chuẩn mẫu của sai số tuyệt đối
std_mae = (
    np.std(errors_abs, ddof=1)
    if len(errors_abs) > 1
    else np.nan
)

# MAPE (loại bỏ các giá trị thực tế bằng 0)
non_zero_mask = y_thucte != 0

mape = (
    np.mean(
        errors_abs[non_zero_mask] /
        np.abs(y_thucte[non_zero_mask])
    ) * 100
    if non_zero_mask.any()
    else np.nan
)

# 5. BẢNG SO SÁNH THỰC TẾ VÀ DỰ BÁO
comparison = pd.DataFrame({
    'Y thực tế': y_thucte,
    'Y dự đoán': y_dudoan,
    'Sai số tuyệt đối': errors_abs,
    'Phần trăm sai số  (%)': np.where(
    y_thucte != 0,
    errors_abs / np.abs(y_thucte) * 100,
    np.nan
)
})

print('\n=== KẾT QUẢ DỰ BÁO TRÊN TẬP TEST ===')
print(comparison.round(2).to_string(index=False))

# 6. In các chỉ số đánh giá
print('\n=== CÁC CHỈ SỐ ĐÁNH GIÁ MÔ HÌNH SES ===')
print(f'MAE     : {mae:.4f}')
print(f'STD MAE : {std_mae:.4f}')
print(f'MAPE    : {mape:.2f}%')

#5. Lưu mô hình

In [ ]:
# Thư mục lưu mô hình
folder = Path("/content/drive/MyDrive/Tân Cảng Hiệp Phước/Train - Model Selection")
folder.mkdir(parents=True, exist_ok=True)

# Đường dẫn lưu mô hình
model_path = folder / "best_model_est.pkl"

# Lưu mô hình đã fit
with open(model_path, "wb") as file:
    pickle.dump(fitted_final, file)

print(f"Đã lưu mô hình ETS tại: {model_path}")